## Primer acercamiento 

Hice un primer acercamiento bastante interesante para saber que es lo que esta pasando o que es lo que tenemos con el DS. 

In [2]:
import pandas as pd

RUTA = '../data/raw/2025O3.xls'

df_crudo = pd.read_excel(RUTA, header=None, nrows=8)
print(df_crudo.to_string())

                    0     1    2    3    4    5    6    7    8    9    10   11   12   13   14   15   16   17   18   19   20   21   22   23   24   25   26   27   28   29   30   31   32   33   34   35   36   37
0                FECHA  HORA  ACO  AJM  AJU  ATI  BJU  CAM  CCA  CHO  COY  CUA  CUT  FAC  FAR  GAM  HGM  INN  IZT  LLA  LPR  MER  MGH  MON  MPA  NEZ  PED  SAC  SAG  SFE  SJA  TAH  TLA  TLI  UAX  UIZ  VIF  XAL
1  2025-01-01 00:00:00     1  -99   46   25   26  -99   12   28   10  -99   29   18   29  -99  -99   17   44   12   23   11   11   31   11  -99    2   34    6  -99  -99  -99   27   19   20    4    7    2  -99
2  2025-01-01 00:00:00     2  -99   47   19   36  -99    7   24    7  -99   28   13   24  -99  -99   25   46   14   11    3   13   23   10  -99    4   29    7  -99  -99  -99   17   22   15    4    8    2  -99
3  2025-01-01 00:00:00     3  -99   47   19   35  -99    2   13    6  -99   22   14    9  -99  -99   14   45    7    3    2   10    4    2  -99    2   26    7  -99 

## Resultados del primer acercamiento 

Al cargar apenas unos datos ya pude saber como esta constituida el csv o el DS. 

Al verlo puedo ver la fecha que constituye de el año, mes y dia, para consecuentemente ver la hora ( del 1 al 24) junto con los datos de cada sitio o sensor que se tiene en la CDMX. 

Inicialmente ya puedo detectar valores interesantes como valores increiblemente bajos como 2 o 1, o hasta valores increimblemente altos o directamente fuera de sentido como el -99.

El -99 implica que indica que:
* El sensor esta fuera de servicio.
* Esta en modo de mantenimiento. 
* No tiene corriente electrica.

## Celda 2

La intencion de esta celda es cuantificar datos, ver que existe y saber cuanto daño hace a nuestro potencial analisis. 

In [3]:
df = pd.read_excel(RUTA, header=0)

print("Dimensiones:", df.shape)
print("\nColumnas:", list(df.columns))
print("\nTipos de dato:")
print(df.dtypes.value_counts())

Dimensiones: (8760, 38)

Columnas: ['FECHA', 'HORA', 'ACO', 'AJM', 'AJU', 'ATI', 'BJU', 'CAM', 'CCA', 'CHO', 'COY', 'CUA', 'CUT', 'FAC', 'FAR', 'GAM', 'HGM', 'INN', 'IZT', 'LLA', 'LPR', 'MER', 'MGH', 'MON', 'MPA', 'NEZ', 'PED', 'SAC', 'SAG', 'SFE', 'SJA', 'TAH', 'TLA', 'TLI', 'UAX', 'UIZ', 'VIF', 'XAL']

Tipos de dato:
int64             37
datetime64[us]     1
Name: count, dtype: int64


## Descubrimientos de la celda 2

Claude me recomendo ver si es que tenemos algun potencial error en los datos o como estan etiquetados. Puedo ver que tengo int64 que potencialmente pueden ser los valores de cada estcion. 

## Celda 3: cuantificar el centinela

En realidad la idea es poder ver todas las estaciones como un solo vector, ademas podermos ver de manera mas facil los valores que tenemos en nuestro DS para poder empezar a trabajar con algunos datos.

In [4]:
estaciones = [c for c in df.columns if c not in ('FECHA', 'HORA')]

valores = df[estaciones].values.ravel()

print("Mínimo absoluto:", valores.min())
print("Máximo absoluto:", valores.max())
print("\nValores más frecuentes:")
print(pd.Series(valores).value_counts().head(5))

Mínimo absoluto: -99
Máximo absoluto: 168

Valores más frecuentes:
-99    96537
 2     10314
 1      7836
 3      7273
 4      5488
Name: count, dtype: int64


## Descubrimientos de la celda 3 

Como se pudo observar esta cuantificacion si tenemos registrados numeros de hasta -99 por el potencial error en los mismo, 

Pero si podemmos detectar la frecuencia y en que tiempo se hizo la muestra y ver si conincide con nuestros conocimiento.

## Celda 4.  El daño concreto

In [6]:
import numpy as np

n_total = valores.size
n_flag  = (valores == -99).sum()

print(f"Celdas totales:   {n_total:,}")
print(f"Celdas con -99:   {n_flag:,}  ({n_flag/n_total*100:.2f}%)")
print()
print(f"Media SIN limpiar: {valores.mean():.2f} ppb")

limpio = np.where(valores == -99, np.nan, valores)
print(f"Media limpia:      {np.nanmean(limpio):.2f} ppb")
print(f"Sesgo introducido: {valores.mean() - np.nanmean(limpio):.2f} ppb")

Celdas totales:   315,360
Celdas con -99:   96,537  (30.61%)

Media SIN limpiar: -7.75 ppb
Media limpia:      32.51 ppb
Sesgo introducido: -40.26 ppb
